# Tabular Foundation Models: a hands-on primer with TabICLv2

[Open in Google Colab](https://colab.research.google.com/github/Affirm/tabular-foundation-models-tutorial/blob/main/materials/notebooks/01_tabicl_primer.ipynb)

**NeurIPS 2026 Education Track.** The reproducible local and CI path uses Python 3.11 with `materials/requirements-lock.txt`; the first cell also supports Colab's managed Python runtime. The submitted archive contains this notebook directly.

### The one idea
> A frozen tabular foundation model can use labeled rows as context and predict new rows without gradient updates on the downstream dataset.

This notebook is a guided illustration, not a benchmark. It uses one fixed context/evaluation split to show the API and compare prediction behavior with a pre-specified XGBoost baseline. The result does not establish that either model is generally superior.

### Data provenance
We use an OpenML compilation uploaded after the named TabICLv2 checkpoint. Some underlying HM Land Registry records may have been public earlier, so upload date alone is not proof against contamination. The relevant evidence is the documented training procedure: the released model was pretrained on generated graph-SCM tasks rather than real benchmark tables.

### What you'll do
1. Load and split one real table into labeled context and evaluation rows.
2. Compare a fixed XGBoost baseline with frozen TabICLv2.
3. Inspect a transparent nearest-row illustration that is **not** model attribution.
4. Generate a small SCM-inspired task to understand synthetic pretraining.

In [ ]:
# Local/CI: install materials/requirements-lock.txt before starting the kernel.
# Colab: this cell installs the tutorial packages into the managed runtime.
import os
import subprocess
import sys

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "tabicl==2.2.0", "openml==0.15.1", "xgboost==2.1.4",
    ])
elif sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        "Use Python 3.11 with materials/requirements-lock.txt, or open this notebook in Colab."
    )

from importlib.metadata import version
from pathlib import Path
import hashlib
import torch
import tabicl  # noqa: F401
from huggingface_hub import hf_hub_download

DEVICE = None  # TabICL auto-selects CUDA, XPU, MPS, then CPU.
N_JOBS = 1
CHECKPOINT = "tabicl-classifier-v2-20260212.ckpt"
CHECKPOINT_REVISION = "4dcd344ece2c00be9e831fdd35bed57b5ad83e19"
CHECKPOINT_SHA256 = "bdc7dbd5e4ff21f8f0456fcf90c6b7cdf72dbea960f2d05b19bec19f9b3d4ed0"
torch.set_num_threads(N_JOBS)

checkpoint_path = Path(hf_hub_download(
    repo_id="jingang/TabICL", filename=CHECKPOINT, revision=CHECKPOINT_REVISION,
))
checkpoint_sha256 = hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()
if checkpoint_sha256 != CHECKPOINT_SHA256:
    raise RuntimeError(
        f"Checkpoint SHA-256 mismatch: expected {CHECKPOINT_SHA256}, got {checkpoint_sha256}"
    )

print("runtime:", "Google Colab" if IN_COLAB else "local/CI", "| Python:", sys.version.split()[0])
for package in ["numpy", "pandas", "scikit-learn", "matplotlib", "xgboost", "openml", "tabicl", "torch"]:
    print(f"{package}: {version(package)}")
print("device selection: automatic | checkpoint:", CHECKPOINT, "| threads:", N_JOBS)
print("checkpoint revision:", CHECKPOINT_REVISION, "| sha256:", checkpoint_sha256)


## 1. A real tabular case study

We use **`essex-house-prices-2025`** ([OpenML dataset 47249](https://www.openml.org/d/47249)), compiled by ValuQ from HM Land Registry Price Paid Data. OpenML records a CC BY license for the compilation; the underlying public records use the UK Open Government Licence v3.0.

Task: predict whether a sale price exceeds the pre-specified threshold **£375,000**. The fixed threshold avoids using evaluation outcomes to define labels. Features are town, postcode district, property type, tenure, new-build flag, and sale month.

We sample 300 labeled context rows from a context pool and keep 100 disjoint rows for evaluation. The 300-row context is the lower boundary of TabICLv2's documented pretraining regime. This single split is chosen for instruction, not statistical comparison across datasets.

In [ ]:
import hashlib
import numpy as np
import pandas as pd
import openml
from sklearn.model_selection import train_test_split

SEED = 42
N_CONTEXT = 300
PRICE_THRESHOLD_GBP = 375_000.0

OPENML_CACHE = Path(".openml-cache").resolve()
openml.config.set_root_cache_directory(OPENML_CACHE)

# OpenML dataset 47249: ValuQ compilation of HM Land Registry Price Paid Data.
ds = openml.datasets.get_dataset(47249, download_data=True, download_qualities=False)
raw, _, _, _ = ds.get_data()

df = pd.DataFrame({
    "town": raw["town"].astype(str),
    "district": raw["postcode_district"].astype(str),
    "property_type": raw["property_type"].astype(str),
    "tenure": raw["tenure"].astype(str),
    "new_build": (raw["new_build"].astype(str) == "yes").astype(int),
    "month": pd.to_datetime(raw["date"]).dt.month.astype(int),
})
price = raw["price_gbp"].astype(float).to_numpy()

CAT_COLS = ["town", "district", "property_type", "tenure"]
NUM_COLS = ["new_build", "month"]
feature_names = CAT_COLS + NUM_COLS
class_names = ["at or below £375k", "above £375k"]

rng = np.random.default_rng(SEED)
sub = rng.choice(len(df), size=1400, replace=False)
X_all = df.iloc[sub].reset_index(drop=True)
y_all = (price[sub] > PRICE_THRESHOLD_GBP).astype(int)

X_pool, X_eval, y_pool, y_eval = train_test_split(
    X_all, y_all, train_size=1300, test_size=100,
    random_state=SEED, stratify=y_all,
)
X_pool = X_pool.reset_index(drop=True)
X_eval = X_eval.reset_index(drop=True)
context_idx, _ = train_test_split(
    np.arange(len(X_pool)), train_size=N_CONTEXT,
    random_state=SEED + 1, stratify=y_pool,
)
X_ctx, y_ctx = X_pool.iloc[context_idx].copy(), y_pool[context_idx]

checksum = hashlib.sha256(pd.util.hash_pandas_object(X_all, index=True).values.tobytes()).hexdigest()
print(f"context: {X_ctx.shape}  evaluation: {X_eval.shape}")
print(f"fixed target threshold: £{int(PRICE_THRESHOLD_GBP):,}  | context balance: {y_ctx.mean():.2f}")
print(f"OpenML dataset id={ds.dataset_id}, version={ds.version}, sampled-data sha256={checksum[:16]}…")

context_preview = X_ctx.head(5).copy()
context_preview["target"] = [class_names[int(value)] for value in y_ctx[:5]]
print("Five labeled context rows:")
display(context_preview)

category_diagnostics = pd.DataFrame({
    "context unique values": [X_ctx[col].nunique() for col in CAT_COLS],
    "evaluation rows unseen in context": [
        int((~X_eval[col].isin(set(X_ctx[col]))).sum()) for col in CAT_COLS
    ],
}, index=CAT_COLS)
print("Categorical coverage for the fixed split:")
display(category_diagnostics)


## 2. Fixed XGBoost baseline

XGBoost receives context-fitted one-hot features; TabICLv2 receives the typed DataFrame through its default ordinal categorical preprocessing. This is an API illustration, not a claim that either preprocessing choice is optimal. The XGBoost settings are pre-specified, not tuned, and evaluation labels are not used to fit either model.

XGBoost runs on CPU with one thread. TabICL automatically selects CUDA, XPU, Apple MPS, or CPU in that order. Accuracy metrics use the same rows, but timing is not presented as a cross-model speed comparison: XGBoost scores a fitted pipeline, while uncached TabICL reprocesses its context during every prediction.

In [ ]:
import time
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, brier_score_loss, roc_curve

xgb = make_pipeline(
    ColumnTransformer([
        ("categorical", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
        ("numeric", "passthrough", NUM_COLS),
    ]),
    XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9,
        eval_metric="logloss", random_state=SEED,
        tree_method="hist", device="cpu", n_jobs=N_JOBS,
    ),
)

t0 = time.perf_counter()
xgb.fit(X_ctx, y_ctx)
xgb_fit_time = time.perf_counter() - t0

t0 = time.perf_counter()
xgb_eval_proba = xgb.predict_proba(X_eval)
xgb_first_predict_time = time.perf_counter() - t0
xgb_end_to_end_time = xgb_fit_time + xgb_first_predict_time

xgb_predict_time = xgb_first_predict_time
xgb_eval_pred = xgb_eval_proba.argmax(1)
xgb_eval_acc = accuracy_score(y_eval, xgb_eval_pred)
xgb_eval_roc_auc = roc_auc_score(y_eval, xgb_eval_proba[:, 1])
xgb_eval_log_loss = log_loss(y_eval, xgb_eval_proba)
xgb_eval_brier = brier_score_loss(y_eval, xgb_eval_proba[:, 1])

print("XGBoost evaluation metrics")
print(f"  accuracy={xgb_eval_acc:.3f}  ROC AUC={xgb_eval_roc_auc:.3f}")
print(f"  log loss={xgb_eval_log_loss:.3f}  Brier score={xgb_eval_brier:.3f}")
print(f"fit={xgb_fit_time:.3f}s  score={xgb_predict_time:.3f}s (single CPU run)")
print(f"fit + first score={xgb_end_to_end_time:.3f}s")


## 3. TabICLv2: frozen in-context prediction

The standard `TabICLClassifier` performs **no downstream gradient updates**. Its `fit()` call validates and preprocesses the table, prepares ensemble views, stores the transformed context, and loads the frozen checkpoint. With `kv_cache=False`, every prediction reprocesses the context. With `kv_cache=True`, `fit()` additionally builds reusable context projections. We measure these as separate inference modes.

One sklearn `predict_proba()` call averages eight shuffled/preprocessed in-context views by default. Conceptually, each view evaluates a PFN-style mapping such as:

```python
logits = model(X_context, y_context, X_query)
```

In [ ]:
from tabicl import TabICLClassifier

def synchronize_accelerator(device):
    """Wait for queued accelerator work so wall-clock timings are meaningful."""
    backend = getattr(torch, torch.device(device).type, None)
    synchronize = getattr(backend, "synchronize", None)
    if synchronize is not None:
        synchronize()

common_tabicl = dict(
    checkpoint_version=CHECKPOINT,
    model_path=str(checkpoint_path),
    allow_auto_download=False,
    n_estimators=8,
    device=DEVICE,
    n_jobs=N_JOBS,
    random_state=SEED,
)

clf_uncached = TabICLClassifier(kv_cache=False, **common_tabicl)
t0 = time.perf_counter()
clf_uncached.fit(X_ctx, y_ctx)
UNCACHED_DEVICE = str(clf_uncached.device_)
synchronize_accelerator(UNCACHED_DEVICE)
tabicl_uncached_fit_time = time.perf_counter() - t0

X_timing = X_eval.iloc[:1]
synchronize_accelerator(UNCACHED_DEVICE)
t0 = time.perf_counter()
tabicl_uncached_proba = clf_uncached.predict_proba(X_timing)
synchronize_accelerator(UNCACHED_DEVICE)
tabicl_uncached_first_predict_time = time.perf_counter() - t0
tabicl_uncached_end_to_end_time = tabicl_uncached_fit_time + tabicl_uncached_first_predict_time

tabicl_uncached_predict_time = tabicl_uncached_first_predict_time

clf_cached = TabICLClassifier(kv_cache=True, **common_tabicl)
synchronize_accelerator(UNCACHED_DEVICE)
t0 = time.perf_counter()
clf_cached.fit(X_ctx, y_ctx)
RESOLVED_DEVICE = str(clf_cached.device_)
synchronize_accelerator(RESOLVED_DEVICE)
tabicl_cached_fit_time = time.perf_counter() - t0

synchronize_accelerator(RESOLVED_DEVICE)
t0 = time.perf_counter()
tabicl_cached_timing_proba = clf_cached.predict_proba(X_timing)
synchronize_accelerator(RESOLVED_DEVICE)
tabicl_cached_first_predict_time = time.perf_counter() - t0
tabicl_cached_end_to_end_time = tabicl_cached_fit_time + tabicl_cached_first_predict_time
tabicl_cached_predict_time = tabicl_cached_first_predict_time

if not np.allclose(tabicl_uncached_proba, tabicl_cached_timing_proba, atol=1e-5, rtol=1e-5):
    raise RuntimeError("Cached and uncached TabICL probabilities diverged")
tabicl_eval_proba = clf_cached.predict_proba(X_eval)
tabicl_eval_pred = tabicl_eval_proba.argmax(1)
tabicl_eval_acc = accuracy_score(y_eval, tabicl_eval_pred)
tabicl_eval_roc_auc = roc_auc_score(y_eval, tabicl_eval_proba[:, 1])
tabicl_eval_log_loss = log_loss(y_eval, tabicl_eval_proba)
tabicl_eval_brier = brier_score_loss(y_eval, tabicl_eval_proba[:, 1])

print("TabICLv2 evaluation metrics")
print(f"  resolved device={RESOLVED_DEVICE}")
print(f"  accuracy={tabicl_eval_acc:.3f}  ROC AUC={tabicl_eval_roc_auc:.3f}")
print(f"  log loss={tabicl_eval_log_loss:.3f}  Brier score={tabicl_eval_brier:.3f}")
print(f"uncached: fit/setup={tabicl_uncached_fit_time:.3f}s, one-row predict={tabicl_uncached_predict_time:.3f}s, end-to-end={tabicl_uncached_end_to_end_time:.3f}s")
print(f"cached: cache-building fit={tabicl_cached_fit_time:.3f}s, one-row predict={tabicl_cached_predict_time:.3f}s, end-to-end={tabicl_cached_end_to_end_time:.3f}s")


## 4. One illustrative comparison

These outputs describe one pre-specified 300-row context and one evaluation partition. They do not establish that either model is generally more accurate or faster. Accuracy and ROC AUC summarize predictions on 100 evaluation rows; the ROC curves show threshold behavior; timing compares TabICL's uncached and cached modes on the same single query.

In [ ]:
import matplotlib.pyplot as plt

comparison = pd.DataFrame([
    {
        "model": "Fixed XGBoost",
        "accuracy": accuracy_score(y_eval, xgb_eval_proba.argmax(1)),
        "roc_auc": roc_auc_score(y_eval, xgb_eval_proba[:, 1]),
        "log_loss": log_loss(y_eval, xgb_eval_proba),
        "brier": brier_score_loss(y_eval, xgb_eval_proba[:, 1]),
    },
    {
        "model": "TabICLv2 (8 views)",
        "accuracy": accuracy_score(y_eval, tabicl_eval_proba.argmax(1)),
        "roc_auc": roc_auc_score(y_eval, tabicl_eval_proba[:, 1]),
        "log_loss": log_loss(y_eval, tabicl_eval_proba),
        "brier": brier_score_loss(y_eval, tabicl_eval_proba[:, 1]),
    },
]).set_index("model")
print("Evaluation metrics on the fixed 100-row split:")
display(comparison.round(4))

prediction_preview = pd.DataFrame({
    "true label": [class_names[int(value)] for value in y_eval[:8]],
    "XGBoost p(above £375k)": xgb_eval_proba[:8, 1],
    "TabICLv2 p(above £375k)": tabicl_eval_proba[:8, 1],
})
print("First eight evaluation predictions:")
display(prediction_preview.round(4))

tabicl_timing = pd.Series({
    "uncached fit/setup": tabicl_uncached_fit_time,
    "uncached prediction (1 row)": tabicl_uncached_predict_time,
    "uncached end-to-end": tabicl_uncached_end_to_end_time,
    "cached fit/cache build": tabicl_cached_fit_time,
    "cached prediction (1 row)": tabicl_cached_predict_time,
    "cached end-to-end": tabicl_cached_end_to_end_time,
}, name="seconds")
display(tabicl_timing.to_frame().round(4))

fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
comparison[["accuracy", "roc_auc"]].plot.bar(
    ax=ax[0], color=["#38bdf8", "#a78bfa"], rot=0,
)
ax[0].set_title("Accuracy and ROC AUC"); ax[0].set_ylim(0, 1); ax[0].set_ylabel("score")

for label, probabilities, color in [
    ("Fixed XGBoost", xgb_eval_proba[:, 1], "#f87171"),
    ("TabICLv2 (8 views)", tabicl_eval_proba[:, 1], "#38bdf8"),
]:
    fpr, tpr, _ = roc_curve(y_eval, probabilities)
    auc = roc_auc_score(y_eval, probabilities)
    ax[1].plot(fpr, tpr, color=color, label=f"{label} (AUC={auc:.3f})")
ax[1].plot([0, 1], [0, 1], color="#94a3b8", linestyle=":")
ax[1].set(xlabel="false positive rate", ylabel="true positive rate", title="ROC curves")
ax[1].legend(fontsize=8)

tabicl_timing[["uncached prediction (1 row)", "cached prediction (1 row)"]].plot.bar(
    ax=ax[2], color=["#a78bfa", "#38bdf8"], rot=0,
)
ax[2].set_title("TabICLv2 inference modes"); ax[2].set_ylabel("seconds (one-row query)")
plt.tight_layout(); plt.show()


## 5. A nearest-row illustration, not a model explanation

The calculation below does **not** inspect TabICLv2 attention, gradients, or predictions, so it is neither attribution nor an explanation of which rows caused the output. It only identifies nearby context rows under a transparent mixed-type distance: Hamming mismatch for categorical/binary fields and cyclic distance for sale month.

In [ ]:
def nearest_row_weights(context, query_row, k=8):
    discrete_cols = CAT_COLS + ["new_build"]
    mismatch = context[discrete_cols].astype(str).ne(query_row[discrete_cols].astype(str)).to_numpy()
    month_delta = np.abs(context["month"].to_numpy() - int(query_row["month"]))
    month_cyclic = np.minimum(month_delta, 12 - month_delta) / 6.0
    distance = (mismatch.sum(axis=1) + month_cyclic) / (len(discrete_cols) + 1)
    weights = np.exp(-5.0 * (distance - distance.min()))
    weights = weights / weights.sum()
    top = np.argsort(-weights)[:k]
    return weights, top, distance

qi = 0
w, top, distance = nearest_row_weights(X_ctx, X_eval.iloc[qi])
print(f"Evaluation row {qi} (true label = {class_names[y_eval[qi]]}):")
for pos in top:
    print(f"  context position {pos:3d}  distance={distance[pos]:.3f}  "
          f"kernel_weight={w[pos]:.3f}  label={class_names[y_ctx[pos]]}")
print("These are nearest rows under the stated distance, not TabICLv2 attention or attribution.")


## 6. A simplified SCM-inspired task

The released TabICLv2 prior generates synthetic datasets from a much richer graph-SCM system: 2–32 latent graph nodes, up to 100 observed features, multiple mechanism families, categorical converters, hidden dimensions, and predictability filters. The small additive-noise SEM below illustrates the idea of sampling a task from a DAG; it is **not** a reproduction or atomic unit of the actual pretraining prior.

In [ ]:
def sample_scm_task(n_rows=300, n_nodes=6, seed=0):
    """Sample a pedagogical additive-noise SEM, not the released TabICLv2 prior."""
    rng = np.random.default_rng(seed)
    values = np.zeros((n_rows, n_nodes))
    dag = {}
    for i in range(n_nodes):
        parents = [j for j in range(i) if rng.random() < 0.5]
        noise = rng.normal(0, 1, n_rows)
        if not parents:
            values[:, i] = noise
            dag[i] = {"parents": [], "function": "root", "weights": []}
        else:
            weights = rng.normal(0, 1, len(parents))
            predictor = values[:, parents] @ weights
            function = rng.choice(["linear", "tanh", "sin"])
            transformed = {
                "linear": predictor,
                "tanh": np.tanh(predictor),
                "sin": np.sin(predictor),
            }[function]
            values[:, i] = transformed + 0.3 * noise
            dag[i] = {"parents": parents, "function": function, "weights": weights.tolist()}

    # Last node has no descendants among the observed features. A fixed threshold
    # preserves row independence and avoids using held-out outcomes to define labels.
    target_col = n_nodes - 1
    target_score = values[:, target_col]
    y = (target_score > 0.0).astype(int)
    feature_cols = list(range(target_col))
    return values[:, feature_cols], y, target_score, target_col, dag

Xs, ys, latent_score, target_col, toy_dag = sample_scm_task(seed=SEED)
print(f"toy task: X={Xs.shape}, target=node {target_col} thresholded at 0, class balance={ys.mean():.2f}")
dag_summary = pd.DataFrame([
    {"node": node, "parents": spec["parents"], "mechanism": spec["function"]}
    for node, spec in toy_dag.items()
])
print("Sampled toy DAG:")
display(dag_summary)
print("Actual pretraining repeatedly samples richer generated tasks, hides query labels, and optimizes classification cross-entropy.")


## Recap

- **Frozen downstream inference:** the standard TabICLv2 estimator performs no gradient updates on this table. `fit()` prepares preprocessing and context; `predict_proba()` runs an eight-view ensemble by default.
- **Documented row regime:** the principal measured example uses 300 context rows, the lower boundary of the official 300 to 48K-row pretraining range.
- **Illustration, not leaderboard:** the comparison uses one dataset and one split to make the inference interface concrete. It does not establish a general winner.
- **Timing scope:** uncached TabICL prediction reprocesses context; cached prediction reuses context projections. The notebook reports both plus their separate end-to-end paths.
- **Nearest rows:** the mixed-type distance is a pedagogical neighbor proxy, not TabICLv2 attention or causal attribution.
- **Generated tasks:** the small SEM illustrates the graph-SCM idea but is not the released TabICLv2 prior.
- **Reproducibility:** the notebook records runtime package versions, immutable checkpoint revision and checksum, device, data version, and seed. Local and CI runs use the complete Python 3.11 lock; Colab uses its managed base runtime plus pinned tutorial packages.

**Next:** use the interactive guide to inspect the architecture and distinguish measured tensors from explanatory simulations.

---
*Sources: [TabICLv2 (ICML 2026)](https://arxiv.org/abs/2602.11139) · [PFNs (Müller et al., ICLR 2022)](https://arxiv.org/abs/2112.10510) · [TabArena (NeurIPS 2025)](https://arxiv.org/abs/2506.16791) · [HM Land Registry Price Paid Data](https://www.gov.uk/government/collections/price-paid-data).*